In [ ]:
import numpy as np
import cvxopt
import scipy

In [ ]:
# def get_qp_tde_inequality_operator_and_data_vector(
#     model: Model, operators: Operators, index: Index
# ) -> tuple[np.ndarray, np.ndarray]:
#     assert index.eigen is not None
#     assert index.tde is not None
#     assert operators.eigen is not None

#     qp_constraint_matrix = np.zeros(
#         (4 * index.n_tde_total, index.n_operator_cols_eigen)
#     )
#     qp_constraint_data_vector = np.zeros(4 * index.n_tde_total)

#     for i in range(index.n_meshes):
#         # TDE strike- and dip-slip lower bounds
#         lower_ss = model.meshes[i].config.elastic_constraints_ss.lower
#         assert lower_ss is not None

#         lower_ds = model.meshes[i].config.elastic_constraints_ds.lower
#         assert lower_ds is not None

#         lower_bound_current_mesh = interleave2(
#             lower_ss * np.ones(index.tde.n_tde[i]),
#             lower_ds * np.ones(index.tde.n_tde[i]),
#         )

#         # TDE strike- and dip-slip upper bounds
#         upper_ss = model.meshes[i].config.elastic_constraints_ss.upper
#         assert upper_ss is not None
#         upper_ds = model.meshes[i].config.elastic_constraints_ds.upper
#         assert upper_ds is not None
#         upper_bound_current_mesh = interleave2(
#             upper_ss * np.ones(index.tde.n_tde[i]),
#             upper_ds * np.ones(index.tde.n_tde[i]),
#         )

#         # Insert TDE lower bounds into QP constraint data vector (note negative sign)
#         qp_constraint_data_vector[
#             index.eigen.qp_constraint_tde_rate_start_row_eigen[
#                 i
#             ] : index.eigen.qp_constraint_tde_rate_start_row_eigen[i]
#             + 2 * index.tde.n_tde[i]
#         ] = -lower_bound_current_mesh

#         # Insert TDE upper bounds into QP constraint data vector
#         qp_constraint_data_vector[
#             index.eigen.qp_constraint_tde_rate_start_row_eigen[i]
#             + 2 * index.tde.n_tde[i] : index.eigen.qp_constraint_tde_rate_end_row_eigen[
#                 i
#             ]
#         ] = upper_bound_current_mesh

#         # Insert eigenmode to TDE slip operator into QP constraint data vector for lower bounds (note negative sign)
#         qp_constraint_matrix[
#             index.eigen.qp_constraint_tde_rate_start_row_eigen[
#                 i
#             ] : index.eigen.qp_constraint_tde_rate_start_row_eigen[i]
#             + 2 * index.tde.n_tde[i],
#             index.eigen.start_col_eigen[i] : index.eigen.end_col_eigen[i],
#         ] = -operators.eigen.eigenvectors_to_tde_slip[i]

#         # Insert eigenmode to TDE slip operator into QP constraint data vector for lower bounds
#         qp_constraint_matrix[
#             index.eigen.qp_constraint_tde_rate_start_row_eigen[i]
#             + 2 * index.tde.n_tde[i] : index.eigen.qp_constraint_tde_rate_end_row_eigen[
#                 i
#             ],
#             index.eigen.start_col_eigen[i] : index.eigen.end_col_eigen[i],
#         ] = operators.eigen.eigenvectors_to_tde_slip[i]

#     return qp_constraint_matrix, qp_constraint_data_vector


In [ ]:
def lsqlin_qp(
    C,
    d,
    reg=0,
    A=None,
    b=None,
    Aeq=None,
    beq=None,
    lb=None,
    ub=None,
    x0=None,
    opts=None,
):
    """Solve linear constrained l2-regularized least squares. Can
    handle both dense and sparse matrices. Call modeled after Matlab's
    lsqlin. It is actually wrapper around CVXOPT QP solver.

        min_x ||C*x  - d||^2_2 + reg * ||x||^2_2
        s.t.  A * x <= b
              Aeq * x = beq
              lb <= x <= ub

    Input arguments:
        C   is m x n dense or sparse matrix
        d   is n x 1 dense matrix
        reg is regularization parameter
        A   is p x n dense or sparse matrix
        b   is p x 1 dense matrix
        Aeq is q x n dense or sparse matrix
        beq is q x 1 dense matrix
        lb  is n x 1 matrix or scalar
        ub  is n x 1 matrix or scalar

    Output arguments:
        Return dictionary, the output of CVXOPT QP.

    Dont pass matlab-like empty lists to avoid setting parameters,
    just use None:
        lsqlin(C, d, 0.05, None, None, Aeq, beq) #Correct
        lsqlin(C, d, 0.05, [], [], Aeq, beq) #Wrong!

    Provenance notes:
    Found a few places on Github:
    - https://github.com/KasparP/PSI_simulations/blob/master/Python/SLAPMi/lsqlin.py
    - https://github.com/geospace-code/airtools/blob/main/src/airtools/lsqlin.py

    Some attribution:
    __author__ = "Valeriy Vishnevskiy", "Michael Hirsch"
    __email__ = "valera.vishnevskiy@yandex.ru"
    __version__ = "1.0"
    __date__ = "22.11.2013"
    __license__ = "MIT"
    """

    # Helper functions
    def scipy_sparse_to_spmatrix(A):
        coo = A.tocoo()
        SP = cvxopt.spmatrix(coo.data, coo.row.tolist(), coo.col.tolist())
        return SP

    def spmatrix_sparse_to_scipy(A):
        data = np.array(A.V).squeeze()
        rows = np.array(A.I).squeeze()
        cols = np.array(A.J).squeeze()
        return scipy.sparse.coo_matrix((data, (rows, cols)))

    def sparse_None_vstack(A1, A2):
        if A1 is None:
            return A2
        else:
            return scipy.sparse.vstack([A1, A2])

    def numpy_None_vstack(A1, A2):
        if A1 is None:
            return A2
        elif isinstance(A1, np.ndarray):
            return np.vstack([A1, A2])
        elif isinstance(A1, cvxopt.spmatrix):
            return np.vstack([cvxopt_to_numpy_matrix(A1).todense(), A2])

    def numpy_None_concatenate(A1, A2):
        if A1 is None:
            return A2
        else:
            return np.concatenate([A1, A2])

    def numpy_to_cvxopt_matrix(A):
        if A is None:
            return

        if scipy.sparse.issparse(A):
            if isinstance(A, scipy.sparse.spmatrix):
                return scipy_sparse_to_spmatrix(A)
            else:
                return A
        else:
            if isinstance(A, np.ndarray):
                if A.ndim == 1:
                    return cvxopt.matrix(A, (A.shape[0], 1), "d")
                else:
                    return cvxopt.matrix(A, A.shape, "d")
            else:
                return A

    def cvxopt_to_numpy_matrix(A):
        if A is None:
            return
        if isinstance(A, cvxopt.spmatrix):
            return spmatrix_sparse_to_scipy(A)
        elif isinstance(A, cvxopt.matrix):
            return np.asarray(A).squeeze()
        else:
            return np.asarray(A).squeeze()

    # Main function body
    if scipy.sparse.issparse(A):  # detects both np and cxopt sparse
        sparse_case = True
        # We need A to be scipy sparse, as I couldn't find how
        # CVXOPT spmatrix can be vstacked
        if isinstance(A, cvxopt.spmatrix):
            A = spmatrix_sparse_to_scipy(A)
    else:
        sparse_case = False

    C = numpy_to_cvxopt_matrix(C)
    d = numpy_to_cvxopt_matrix(d)
    Q = C.T * C
    q = -d.T * C
    nvars = C.size[1]

    if reg > 0:
        if sparse_case:
            i = scipy_sparse_to_spmatrix(scipy.sparse.eye(nvars, nvars, format="coo"))
        else:
            i = cvxopt.matrix(np.eye(nvars), (nvars, nvars), "d")
        Q = Q + reg * i

    lb = cvxopt_to_numpy_matrix(lb)
    ub = cvxopt_to_numpy_matrix(ub)
    b = cvxopt_to_numpy_matrix(b)

    if lb is not None:  # Modify 'A' and 'b' to add lb inequalities
        if lb.size == 1:
            lb = np.repeat(lb, nvars)

        if sparse_case:
            lb_A = -scipy.sparse.eye(nvars, nvars, format="coo")
            A = sparse_None_vstack(A, lb_A)
        else:
            lb_A = -np.eye(nvars)
            A = numpy_None_vstack(A, lb_A)
        b = numpy_None_concatenate(b, -lb)
    if ub is not None:  # Modify 'A' and 'b' to add ub inequalities
        if ub.size == 1:
            ub = np.repeat(ub, nvars)
        if sparse_case:
            ub_A = scipy.sparse.eye(nvars, nvars, format="coo")
            A = sparse_None_vstack(A, ub_A)
        else:
            ub_A = np.eye(nvars)
            A = numpy_None_vstack(A, ub_A)
        b = numpy_None_concatenate(b, ub)

    # Convert data to CVXOPT format
    A = numpy_to_cvxopt_matrix(A)
    Aeq = numpy_to_cvxopt_matrix(Aeq)
    b = numpy_to_cvxopt_matrix(b)
    beq = numpy_to_cvxopt_matrix(beq)

    # Set up options
    if opts is not None:
        for k, v in opts.items():
            cvxopt.solvers.options[k] = v

    # Run CVXOPT.SQP solver
    sol = cvxopt.solvers.qp(Q, q.T, A, b, Aeq, beq, None, x0)
    return sol



In [ ]:
# Get QP bounds as inequality constraints
# qp_inequality_constraints_matrix, qp_inequality_constraints_data_vector = (
#     celeri.get_qp_all_inequality_operator_and_data_vector(
#         index, meshes, operators, segment, block
#     )
# )

design_matrix = []
data_vector = []
weighting_vector = []
qp_inequality_constraints_matrix = []
qp_inequality_constraints_data_vector = []

# Slip bounded QP solve
opts = {"show_progress": True}
solution_qp = lsqlin_qp(
    design_matrix * np.sqrt(weighting_vector[:, None]),
    data_vector * np.sqrt(weighting_vector),
    0,
    qp_inequality_constraints_matrix,  # Inequality matrix
    qp_inequality_constraints_data_vector,  # Inequality data vector
    None,
    None,
    None,
    None,
    None,
    opts,
)